In [1]:
import os
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("output", exist_ok=True)

dataset_text = """
machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.

long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.

language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.

generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance.
"""

sentences = [s.strip() for s in dataset_text.split("\n") if s.strip()]

with open("output/dataset.txt","w") as f:
    f.write(dataset_text)

In [2]:
tokens = []
for s in sentences:
    tokens.extend(s.split())

vocab = sorted(set(tokens))
word2idx = {w:i+1 for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

vocab_size = len(word2idx) + 1
with open("output/vocab.txt","w") as f:
    for w in vocab:
        f.write(w+"\n")

In [3]:
SEQ_LEN = 3

X = []
Y = []
for s in sentences:
    words = s.split()
    for i in range(len(words)-SEQ_LEN):
        seq = words[i:i+SEQ_LEN]
        target = words[i+SEQ_LEN]
        X.append([word2idx[w] for w in seq])
        Y.append(word2idx[target])

class SeqDataset(Dataset):
    def __init__(self,X,Y):
        self.X=torch.tensor(X)
        self.Y=torch.tensor(Y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,i):
        return self.X[i],self.Y[i]

dataset = SeqDataset(X,Y)
loader = DataLoader(dataset,batch_size=16,shuffle=True)

In [4]:
class LSTMGenerator(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size,64)
        self.lstm = nn.LSTM(64,128,batch_first=True)
        self.fc = nn.Linear(128,vocab_size)

    def forward(self,x):
        x = self.embed(x)
        out,_ = self.lstm(x)
        out = out[:,-1,:]
        out = self.fc(out)
        return out

model = LSTMGenerator(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

EPOCHS = 500
loss_history=[]

for epoch in range(EPOCHS):
    total_loss = 0
    for x,y in loader:
        x=x.to(device)
        y=y.to(device)
        pred = model(x)
        loss = criterion(pred,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss/len(loader)
    print(f"Epoch {epoch+1} Loss {avg_loss:.4f}")
    loss_history.append(avg_loss)

torch.save(model.state_dict(),"output/lstm_model.pt")

with open("output/lstm_training_log.txt","w") as f:
    for i,l in enumerate(loss_history):
        f.write(f"Epoch {i+1} Loss {l}\n")

Epoch 1 Loss 4.4918
Epoch 2 Loss 4.4071
Epoch 3 Loss 4.3347
Epoch 4 Loss 4.2613
Epoch 5 Loss 4.1829
Epoch 6 Loss 4.0976
Epoch 7 Loss 4.0041
Epoch 8 Loss 3.8955
Epoch 9 Loss 3.7721
Epoch 10 Loss 3.6298
Epoch 11 Loss 3.4634
Epoch 12 Loss 3.2715
Epoch 13 Loss 3.0534
Epoch 14 Loss 2.8106
Epoch 15 Loss 2.5472
Epoch 16 Loss 2.2730
Epoch 17 Loss 1.9943
Epoch 18 Loss 1.7232
Epoch 19 Loss 1.4608
Epoch 20 Loss 1.2198
Epoch 21 Loss 1.0075
Epoch 22 Loss 0.8258
Epoch 23 Loss 0.6746
Epoch 24 Loss 0.5516
Epoch 25 Loss 0.4550
Epoch 26 Loss 0.3790
Epoch 27 Loss 0.3203
Epoch 28 Loss 0.2733
Epoch 29 Loss 0.2356
Epoch 30 Loss 0.2055
Epoch 31 Loss 0.1815
Epoch 32 Loss 0.1616
Epoch 33 Loss 0.1455
Epoch 34 Loss 0.1318
Epoch 35 Loss 0.1205
Epoch 36 Loss 0.1105
Epoch 37 Loss 0.1020
Epoch 38 Loss 0.0946
Epoch 39 Loss 0.0882
Epoch 40 Loss 0.0823
Epoch 41 Loss 0.0771
Epoch 42 Loss 0.0727
Epoch 43 Loss 0.0686
Epoch 44 Loss 0.0648
Epoch 45 Loss 0.0613
Epoch 46 Loss 0.0583
Epoch 47 Loss 0.0555
Epoch 48 Loss 0.0529
E

In [5]:
def generate(model,seed,length=10):
    model.eval()
    words = seed.split()
    for _ in range(length):
        seq = words[-SEQ_LEN:]
        seq = [word2idx.get(w,0) for w in seq]
        x = torch.tensor([seq]).to(device)
        with torch.no_grad():
            pred = model(x)
        prob = torch.softmax(pred, dim=-1)
        idx = torch.multinomial(prob,1).item()
        next_word = idx2word.get(idx,"")
        words.append(next_word)
    return " ".join(words)

samples=[]
seed = "machine learning models"

for i in range(5):
    g = generate(model,seed,8)
    print(g)
    samples.append(g)

with open("output/lstm_generated_sequences.txt","w") as f:
    for s in samples:
        f.write(s+"\n")

machine learning models learn patterns from data. melodies. time steps. states
machine learning models learn patterns from data. data. time designed steps.
machine learning models learn patterns from data. data. time deep useful
machine learning models learn patterns from data. data. widely to flow.
machine learning models learn patterns from data. data. distributions. time steps.


In [6]:
class TransformerGenerator(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size,64)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )
        self.fc = nn.Linear(64,vocab_size)

    def forward(self,x):
        x = self.embed(x)
        out = self.transformer(x)
        out = out[:,-1]
        out = self.fc(out)
        return out

model_t = TransformerGenerator(vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_t.parameters(),lr=0.001)

loss_history_t=[]

for epoch in range(EPOCHS):
    total_loss=0
    for x,y in loader:
        x=x.to(device)
        y=y.to(device)
        pred = model_t(x)
        loss = criterion(pred,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()

    avg_loss = total_loss/len(loader)
    print(f"Epoch {epoch+1} Loss {avg_loss:.4f}")
    loss_history_t.append(avg_loss)

torch.save(model_t.state_dict(),"output/transformer_model.pt")

Epoch 1 Loss 4.8597
Epoch 2 Loss 3.7144
Epoch 3 Loss 2.9382
Epoch 4 Loss 2.3400
Epoch 5 Loss 1.9476
Epoch 6 Loss 1.6072
Epoch 7 Loss 1.4089
Epoch 8 Loss 1.2298
Epoch 9 Loss 1.0607
Epoch 10 Loss 0.9466
Epoch 11 Loss 0.8345
Epoch 12 Loss 0.7438
Epoch 13 Loss 0.6807
Epoch 14 Loss 0.5899
Epoch 15 Loss 0.5476
Epoch 16 Loss 0.5025
Epoch 17 Loss 0.4443
Epoch 18 Loss 0.4059
Epoch 19 Loss 0.3697
Epoch 20 Loss 0.3562
Epoch 21 Loss 0.3032
Epoch 22 Loss 0.2903
Epoch 23 Loss 0.2738
Epoch 24 Loss 0.2512
Epoch 25 Loss 0.2362
Epoch 26 Loss 0.2248
Epoch 27 Loss 0.2068
Epoch 28 Loss 0.1917
Epoch 29 Loss 0.1828
Epoch 30 Loss 0.1714
Epoch 31 Loss 0.1611
Epoch 32 Loss 0.1565
Epoch 33 Loss 0.1467
Epoch 34 Loss 0.1359
Epoch 35 Loss 0.1328
Epoch 36 Loss 0.1241
Epoch 37 Loss 0.1205
Epoch 38 Loss 0.1132
Epoch 39 Loss 0.1114
Epoch 40 Loss 0.1080
Epoch 41 Loss 0.1075
Epoch 42 Loss 0.0958
Epoch 43 Loss 0.0928
Epoch 44 Loss 0.0900
Epoch 45 Loss 0.0854
Epoch 46 Loss 0.0869
Epoch 47 Loss 0.0808
Epoch 48 Loss 0.0778
E

In [7]:
samples_t=[]
for i in range(5):
    g = generate(model_t,"sequence models process",8)
    print(g)
    samples_t.append(g)

with open("output/transformer_generated_sequences.txt","w") as f:
    for s in samples_t:
        f.write(s+"\n")

sequence models process data step by step. many applications. data. patterns
sequence models process data step by step. intelligence. predicts future values.
sequence models process data step by step. used in artificial intelligence.
sequence models process data step by step. many applications. by step.
sequence models process data step by step. flow. information flow. networks


In [8]:
with open("output/evaluation.txt","w") as f:
    f.write("Sequence Generation Lab\n\n")
    f.write("Model 1 : LSTM\n")
    f.write("Model 2 : Transformer\n\n")
    f.write("Generated samples saved.\n")

zip_path="lab9_output.zip"
with zipfile.ZipFile(zip_path,"w") as zipf:
    for root,dirs,files in os.walk("output"):
        for file in files:
            path=os.path.join(root,file)
            zipf.write(path)

print("Zip created:",zip_path)

from google.colab import files
files.download("lab9_output.zip")

Zip created: lab9_output.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>